# Predictive Modelling

This notebook builds predictive models on the cleaned country-year dataset produced by `I-cleaning.ipynb` and explored in `II-analysis.ipynb`.

**Not yet started.** The section below records constraints that were measured on the cleaned dataset during the cleaning and analysis stages. They are written down here because each one produces a *plausible-looking* result rather than an error, and would otherwise be discovered only after the modelling work had been built on top of them.

---

<table align="left">
  <tr>
    <td>
      <a href="https://colab.research.google.com/github/ValentimPiazera/wb-economic-freedom/blob/main/notebooks/III-ml.ipynb" target="_parent">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
      </a>
    </td>
    <td>
      <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ValentimPiazera/wb-economic-freedom/blob/main/notebooks/III-ml.ipynb">
        <img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/>
      </a>
    </td>
  </tr>
</table>

> **Note:** this notebook is written to run from the **repository root**, not from `notebooks/`.
> Locally, `.vscode/settings.json` already points VS Code's Jupyter root at the workspace folder;
> from a terminal, launch Jupyter from the repository root.
>
> If you're running on Google Colab or Kaggle, run the cell below first: it clones the repository
> so that the `src/` module and data files are available.
>
> On **Kaggle**, make sure Internet access is enabled: *Settings → Internet → On*.

In [ ]:
# Setup: clone repo when running on Colab/Kaggle (skip if running locally!)
import os

IN_COLAB_OR_KAGGLE = (
    "google.colab" in str(get_ipython())
    or "kaggle" in os.environ.get("KAGGLE_URL_BASE", "").lower()
)

if IN_COLAB_OR_KAGGLE:
    if not os.path.exists("wb-economic-freedom"):
        !git clone https://github.com/ValentimPiazera/wb-economic-freedom.git
    # The repo root is the working directory, so `from src import viz`
    # and the `data/` paths below resolve the same way as they do locally.
    %cd wb-economic-freedom

## Modelling Constraints

Every figure below was measured on `data/processed/wb_economic_freedom_merged.csv`, not estimated.

### 1. `GDP per capita` is an exact function of two other columns

`GDP per capita (current US$)` is `GDP (current US$) / Population, total` — maximum relative error between the two, across all 4,581 rows, is `4.0e-5` (median `4.1e-7`). Before Part VIII of `I-cleaning.ipynb` rounded the export to a readable precision that error was `2.5e-16`, so what is left is the rounding rather than any real gap. It is the identity, not a correlation.

If GDP per capita is the target, `GDP (current US$)` and `Population, total` must be removed from the feature matrix. Otherwise the model reconstructs the target exactly, reports R² ≈ 1.00, and looks like a success.

### 2. `Overall Score` is very nearly a function of its own sub-components

The Heritage Overall Score is the average of its components. Even with `Fiscal Health` and `Judicial Effectiveness` dropped during cleaning, the **simple mean of the ten remaining components** predicts the Overall Score with **R² = 0.976** (correlation 0.988).

Predicting `Overall Score` from `Property Rights`, `Tax Burden`, `Trade Freedom` and the rest is therefore re-deriving Heritage's own formula, not learning anything about the world. If the Overall Score is the target, its components cannot be features.

### 3. A random train/test split leaks through autocorrelation

Country-year rows are close to constant from one year to the next:

| Column | corr(t, t−1) |
| --- | --- |
| Overall Score | 0.984 |
| GDP per capita | 0.993 |
| Life expectancy | 0.993 |

`train_test_split(shuffle=True)` puts Portugal-2014 in train and Portugal-2015 in test, so the model memorises the country rather than the relationship, and the validation score is inflated. Use `GroupKFold(groups=df["Country Name"])`, a temporal split (train ≤ 2019, test ≥ 2020), or both.

### 4. A naive `dropna()` silently deletes 48% of the data and whole years

`df.dropna()` reduces the dataset from 4,581 rows to **2,385 (52%)**, and the surviving years are **2005–2024 only**:

- 2001–2004 disappear with `Labor Freedom`, which entered the Index methodology later.
- **2025 disappears entirely**, because `Life expectancy` has 0 of 183 values in that year.

No warning is raised. A project documented as covering 25 years would quietly be modelling 20.

### 5. Missingness is informative, so imputation injects the inverse bias

Missingness correlates with the outcome being predicted:

| | Median GDP per capita |
| --- | --- |
| Countries **with** secondary school enrollment | $5,801 |
| Countries **without** it | $2,510 |

Mean or median imputation pulls the poorest countries towards rich-country values, erasing the signal the model is meant to find. The mirror image is also a trap: gradient-boosted trees handle `NaN` natively and will learn "missing → poor country", which works but means part of the model's accuracy comes from the quality of a country's statistical agency rather than from economics.

Either treatment is defensible; not choosing is not. If the missingness signal is wanted, add explicit indicator columns instead of leaving it implicit.

### 6. The `Index Year` lag stops being cosmetic

As documented in Part IV of `I-cleaning.ipynb`, the Heritage Index published for year *N* is graded on data collected through roughly mid-*N-1*, while the World Bank indicators for year *N* describe calendar year *N*. For descriptive analysis this is immaterial; for any causal framing ("does economic freedom raise growth?") the direction and size of that offset is what carries the claim.

### Also carried over from cleaning

- `Foreign direct investment, net inflows (% of GDP)` concentrates its extremes in four financial-centre economies (Liechtenstein, Malta, Cyprus, Luxembourg). With `StandardScaler` and a linear model, those rows dominate the fit; consider winsorising, log-scaling, or a `financial_centre` flag.
- `Labor Freedom` is ~19.5% missing before 2009, and `School enrollment, secondary` never exceeds ~71% coverage in any year.

---